# HKI + GCP: Full Stack Integration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/h3nok/HKI/blob/main/notebooks/05_gcp_integration.ipynb)

This notebook covers HKI enforcement across the full GCP agentic stack:

| Section | GCP Service | HKI Primitive |
|---------|-------------|---------------|
| 1 | Auth + envelope factory | `validate_envelope` |
| 2 | Vertex AI / Gemini via LiteLLM | `HkiLiteLLMCallback`, `derive_hki_cache_key` |
| 3 | AlloyDB + pgvector | `assert_artifact_visible`, domain-scoped SQL |
| 4 | Cloud Pub/Sub async jobs | Envelope in message body (HKI-T04 fix) |
| 5 | Google ADK agent loop | `HkiBeforeAgentCallback`, `HkiBeforeToolCallback` |
| 6 | Document AI + GCS ingestion | Domain-labelled artifact storage |
| 7 | End-to-end: gateway → ADK → retrieval → LLM | All primitives combined |

**GCP credentials:** Each section degrades to a documented stub when credentials
are not present, so you can follow the pattern without a live project.

**Prerequisites:** `GOOGLE_CLOUD_PROJECT` env var or Colab's GCP auth.

In [ ]:
%pip install hki-runtime hki-langchain hki-litellm hki-adk -q
%pip install google-genai google-cloud-aiplatform google-cloud-pubsub \
             google-cloud-storage asyncpg litellm -q

In [ ]:
import os, json, time, base64, asyncio
import hki_runtime

# ── Project config ────────────────────────────────────────────────────────────
GCP_PROJECT  = os.environ.get("GOOGLE_CLOUD_PROJECT", "your-project-id")
GCP_LOCATION = os.environ.get("GCP_LOCATION", "us-central1")
LIVE_GCP     = GCP_PROJECT != "your-project-id"  # True when real credentials present

if LIVE_GCP:
    print(f"Live GCP project: {GCP_PROJECT} / {GCP_LOCATION}")
else:
    print("No GCP credentials found — stubs will be used. Patterns are identical.")

# ── Shared envelope factory ───────────────────────────────────────────────────
def make_envelope(domain: str, purpose: str = "retrieve", org: str = "org_acme") -> dict:
    return {
        "hki_version": "1.0",
        "envelope_id": f"env_{domain}_{int(time.time())}",
        "org_id": org,
        "subject_id": "user_42",
        "active_domain": domain,
        "authorized_domains": [domain],
        "purpose": purpose,
        "risk_tier": "read-only",
        "policy_pack_id": f"{domain}@2026-05",
        "issued_at": 0,
        "expires_at": 9_999_999_999,
        "issuer": "gateway.acme.internal",
        "signature": "ed25519:placeholder",  # replace with real Ed25519 sig at gateway
    }

def validated(domain: str, **kw) -> hki_runtime.HkiEnvelope:
    result = hki_runtime.validate_envelope(make_envelope(domain, **kw))
    assert result.ok, result.issues
    return result.envelope

print("Setup complete")

---
## Section 1 — Envelope at the gateway boundary

In production the gateway (FastAPI BFF or Cloud Run service) mints and signs
the envelope from the authenticated session. All downstream services receive
it as a header and call `validate_envelope` before processing anything.

```
Browser / API client
    │  POST /chat  {"domain": "pharmacy", "message": "..."}
    ▼
HKI Gateway (Cloud Run)
    │  mint envelope → sign Ed25519 → attach as X-HKI-Envelope header
    ▼
Orchestrator service  ←  HkiMiddleware validates envelope on every request
    │
    ▼
Knowledge API  ←  asserts artifact.domain == envelope.active_domain
```

In [ ]:
from hki_runtime.fastapi import HkiMiddleware
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient

gateway = FastAPI(title="HKI Gateway")
gateway.add_middleware(HkiMiddleware)

@gateway.post("/chat")
async def chat(request: Request, body: dict):
    env = request.state.hki
    return {
        "active_domain": env.active_domain,
        "envelope_id":   env.envelope_id,
        "message":       body.get("message"),
    }

client = TestClient(gateway, raise_server_exceptions=False)

# Encode the envelope as the middleware expects (base64-encoded JSON in header)
def encode_envelope(env: dict) -> str:
    return base64.b64encode(json.dumps(env).encode()).decode()

r = client.post(
    "/chat",
    json={"message": "What are the refund rules?"},
    headers={"X-HKI-Envelope": encode_envelope(make_envelope("pharmacy"))},
)
print(f"[{r.status_code}] {r.json()}")

# Missing envelope → 401
r2 = client.post("/chat", json={"message": "sneaky request"})
print(f"[{r2.status_code}] no envelope → blocked")

---
## Section 2 — Vertex AI / Gemini with HkiLiteLLMCallback

LiteLLM is the LLM proxy used in the orchestrator service. `HkiLiteLLMCallback`
hooks into every LiteLLM call:
- Validates the envelope before the API call is made
- Stamps a domain-bound cache key into `metadata.hki_cache_key`
- Rejects calls that arrive without an envelope
- Attaches HKI attributes to any observability span

The model string for Vertex AI via LiteLLM is `vertex_ai/<model>`, e.g.
`vertex_ai/gemini-2.0-flash`.

In [ ]:
import hki_litellm
import litellm

# Register the HKI callback globally — every litellm.completion() call goes through it
litellm.callbacks = [hki_litellm.HkiLiteLLMCallback()]
litellm.set_verbose = False

PHARMACY_ENV = make_envelope("pharmacy", purpose="chat")

# ── Pattern: call kwargs dict ─────────────────────────────────────────────────
# hki_litellm.pre_call() is what HkiLiteLLMCallback calls internally.
# You can also use it directly to inspect what gets stamped.

kwargs = {
    "hki_envelope": PHARMACY_ENV,
    "model":        "vertex_ai/gemini-2.0-flash",
    "messages":     [{"role": "user", "content": "What is the prescription pickup policy?"}],
    "metadata":     {},
}

envelope_result = hki_litellm.pre_call(kwargs)

print(f"Active domain:   {envelope_result.active_domain}")
print(f"Cache key:       {kwargs['metadata']['hki_cache_key'][:40]}...")
print(f"Model routing:   {kwargs['model']}")
print()

# ── What a real LiteLLM call looks like (requires GCP credentials) ────────────
if LIVE_GCP:
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["VERTEXAI_PROJECT"]  = GCP_PROJECT
    os.environ["VERTEXAI_LOCATION"] = GCP_LOCATION

    response = litellm.completion(
        model="vertex_ai/gemini-2.0-flash",
        messages=[{"role": "user", "content": "Summarise the HKI standard in one sentence."}],
        metadata={"hki_envelope": PHARMACY_ENV},
    )
    print(response.choices[0].message.content)
else:
    print("[stub] Without GCP credentials, litellm.completion() would call:")
    print(f"  model=vertex_ai/gemini-2.0-flash")
    print(f"  metadata.hki_envelope.active_domain={PHARMACY_ENV['active_domain']}")
    print(f"  metadata.hki_cache_key={kwargs['metadata']['hki_cache_key'][:30]}...")

In [ ]:
# ── Rejected: call without envelope ──────────────────────────────────────────
try:
    bad_kwargs = {
        "model":    "vertex_ai/gemini-2.0-flash",
        "messages": [{"role": "user", "content": "leak something"}],
        "metadata": {},
        # no hki_envelope key
    }
    hki_litellm.pre_call(bad_kwargs)
    print("ERROR: should have been blocked")
except Exception as e:
    print(f"No envelope → blocked: {type(e).__name__}: {e}")

# ── Rejected: cross-domain cache key collision is impossible ─────────────────
travel_key   = hki_runtime.derive_hki_cache_key({"envelope": make_envelope("travel"),   "operation": "chat.completion", "input": {"query": "refund"}})
pharmacy_key = hki_runtime.derive_hki_cache_key({"envelope": make_envelope("pharmacy"), "operation": "chat.completion", "input": {"query": "refund"}})
print(f"\nSame query, different domains → different cache keys: {travel_key[:20]}... ≠ {pharmacy_key[:20]}...")

---
## Section 3 — AlloyDB + pgvector: domain-scoped vector search

The knowledge API uses AlloyDB (PostgreSQL-compatible) with pgvector for hybrid
vector + BM25 search. Every row carries a `domain` column. HKI enforces that:

1. The SQL query always includes `WHERE domain = $active_domain`
2. Every returned row is post-filtered through `assert_artifact_visible`

This gives defence in depth: the SQL filter is fast; the HKI check catches
any ORM/query-builder that accidentally drops the WHERE clause.

In [ ]:
# ── Schema (matches knowledge-api production schema) ─────────────────────────
"""
CREATE TABLE knowledge_chunks (
    id           UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    org_id       TEXT NOT NULL,
    domain       TEXT NOT NULL,          -- HKI domain label
    content      TEXT NOT NULL,
    embedding    vector(768),
    metadata     JSONB DEFAULT '{}',
    created_at   TIMESTAMPTZ DEFAULT NOW()
);

-- HKI-required index: every domain query must hit this
CREATE INDEX ON knowledge_chunks (org_id, domain);

-- HNSW index for ANN search
CREATE INDEX ON knowledge_chunks USING hnsw (embedding vector_cosine_ops);
"""
print("Schema above matches the production AlloyDB table")

In [ ]:
# ── HKI-safe vector search function ──────────────────────────────────────────
# This is the pattern used in knowledge-api/src/vector_store.py

import hki_runtime
from typing import Any

async def hki_vector_search(
    pool,                          # asyncpg.Pool
    envelope: hki_runtime.HkiEnvelope,
    query_embedding: list[float],
    top_k: int = 5,
) -> list[dict]:
    """
    Domain-isolated vector search.
    - SQL WHERE clause binds org_id and domain from the signed envelope
    - Every returned row is post-verified with assert_artifact_visible
    """
    SQL = """
        SELECT id, org_id, domain, content, metadata,
               1 - (embedding <=> $1::vector) AS score
        FROM   knowledge_chunks
        WHERE  org_id = $2          -- bound from envelope
          AND  domain = $3          -- bound from envelope (NOT from request body)
        ORDER  BY embedding <=> $1::vector
        LIMIT  $4
    """
    rows = await pool.fetch(
        SQL,
        query_embedding,
        envelope.org_id,         # from signed envelope — cannot be overridden
        envelope.active_domain,  # from signed envelope
        top_k,
    )

    # Defence in depth: verify every returned row
    results = []
    for row in rows:
        issue = hki_runtime.assert_artifact_visible(envelope, {
            "org_id":        row["org_id"],
            "domain":        row["domain"],
            "artifact_type": "chunk",
            "artifact_id":   str(row["id"]),
        })
        if issue is None:
            results.append(dict(row))
        # rows that fail visibility are silently dropped — they should not exist
        # in this result set (SQL WHERE guarantees it), but we verify anyway

    return results

print("hki_vector_search defined")

In [ ]:
# ── Demonstrate with an in-memory stub ───────────────────────────────────────
# The stub mimics the asyncpg pool interface so the logic above runs identically.

CHUNK_STORE = [
    {"id": "c1", "org_id": "org_acme", "domain": "pharmacy", "content": "Rx pickup: Mon-Fri 9am-6pm", "metadata": {}},
    {"id": "c2", "org_id": "org_acme", "domain": "pharmacy", "content": "Return policy: 30 days with receipt", "metadata": {}},
    {"id": "c3", "org_id": "org_acme", "domain": "travel",   "content": "Hotel cancel: 48h notice",  "metadata": {}},
    {"id": "c4", "org_id": "org_acme", "domain": "travel",   "content": "Flight rebooking fee: $150", "metadata": {}},
]

class _StubPool:
    async def fetch(self, sql, embedding, org_id, domain, top_k):
        # Simulate the WHERE org_id=$2 AND domain=$3 filter
        return [
            r for r in CHUNK_STORE
            if r["org_id"] == org_id and r["domain"] == domain
        ][:top_k]

async def demo_vector_search():
    pool = _StubPool()
    env  = validated("pharmacy")

    chunks = await hki_vector_search(pool, env, query_embedding=[0.1, 0.2], top_k=10)

    print(f"Pharmacy envelope → {len(chunks)} chunks returned:")
    for c in chunks:
        print(f"  [{c['domain']}] {c['content']}")

    travel_env = validated("travel")
    travel_chunks = await hki_vector_search(pool, travel_env, query_embedding=[0.1, 0.2])
    print(f"\nTravel envelope → {len(travel_chunks)} chunks:")
    for c in travel_chunks:
        print(f"  [{c['domain']}] {c['content']}")

asyncio.run(demo_vector_search())

In [ ]:
# ── Live AlloyDB connection (requires ALLOYDB_URL env var) ───────────────────
if os.environ.get("ALLOYDB_URL"):
    import asyncpg

    async def live_demo():
        pool = await asyncpg.create_pool(os.environ["ALLOYDB_URL"], min_size=1, max_size=5)
        env  = validated("pharmacy")

        # embed the query — replace with real Vertex AI embedding client
        # from google import genai
        # client = genai.Client()
        # embedding = client.models.embed_content(model="text-embedding-004", content="pickup hours")
        query_embedding = [0.0] * 768  # placeholder

        results = await hki_vector_search(pool, env, query_embedding)
        print(f"Live results: {len(results)}")
        await pool.close()

    asyncio.run(live_demo())
else:
    print("[stub] Set ALLOYDB_URL=postgresql://... to run against live AlloyDB")
    print("  Pattern: asyncpg.create_pool(dsn) → hki_vector_search(pool, envelope, embedding)")

---
## Section 4 — Cloud Pub/Sub: envelope in async job messages

This is the fix for **HKI-T04** (async job loses domain on resume) in production.

The ingestion pipeline enqueues documents via Pub/Sub. The envelope **must travel
in the Pub/Sub message** — not just in the HTTP context that triggered the upload.
The worker re-validates the envelope on every message pull.

```
Upload endpoint  →  enqueue_with_envelope(payload, envelope)  →  Pub/Sub topic
                                                                        │
                                                          Worker pulls message
                                                          validate_envelope()  ← re-validates
                                                          store chunk with domain label
```

In [ ]:
import json

# ── Message schema (matches ingestion-pipeline-service/src/worker.py) ─────────
def build_ingestion_message(payload: dict, envelope: dict) -> bytes:
    """
    Serialise the job payload with the HKI envelope embedded.
    This is what PubSubPublisher.publish() sends in production.
    """
    message = {
        "envelope": envelope,       # travels with every job
        "payload":  payload,
        "version":  "1.0",
    }
    return json.dumps(message).encode()

def worker_process_message(raw: bytes) -> dict:
    """
    What the Pub/Sub worker does on every message pull.
    Mirrors ingestion-pipeline-service/src/worker.py:process_message()
    """
    message = json.loads(raw)

    # Step 1: re-validate the envelope — always, regardless of source
    result = hki_runtime.validate_envelope(message["envelope"], require_signature=True)
    if not result.ok:
        raise PermissionError(f"invalid job envelope: {[i.message for i in result.issues]}")

    env = result.envelope

    # Step 2: extract domain from validated envelope — NOT from payload
    domain = env.active_domain
    org_id = env.org_id

    # Step 3: store chunk with correct domain label
    chunk = {
        "org_id": org_id,
        "domain": domain,           # from envelope — tamper-proof
        "content": message["payload"]["text"],
        "envelope_id": env.envelope_id,
    }
    return chunk

# ── Simulate enqueue + worker ─────────────────────────────────────────────────
raw = build_ingestion_message(
    payload={"text": "Pharmacy: medication storage guidance"},
    envelope=make_envelope("pharmacy", purpose="ingest"),
)

chunk = worker_process_message(raw)
print(f"Chunk stored with domain: {chunk['domain']}")
print(f"Chunk: {chunk}")

In [ ]:
# ── What happens if someone tampers with the domain in the payload ─────────────
tampered = json.loads(raw)
tampered["payload"]["domain_override"] = "travel"  # attacker tries to mislabel

chunk2 = worker_process_message(json.dumps(tampered).encode())
print(f"After tampering attempt, domain is still: {chunk2['domain']}")
print("(domain comes from the signed envelope — payload field is ignored)")

In [ ]:
# ── Live Pub/Sub (requires google-cloud-pubsub + GCP credentials) ─────────────
PUBSUB_TOPIC = os.environ.get("PUBSUB_TOPIC", "")

if LIVE_GCP and PUBSUB_TOPIC:
    from google.cloud import pubsub_v1

    publisher = pubsub_v1.PublisherClient()
    topic_path = publisher.topic_path(GCP_PROJECT, PUBSUB_TOPIC)

    msg_bytes = build_ingestion_message(
        payload={"text": "Live test document"},
        envelope=make_envelope("pharmacy", purpose="ingest"),
    )
    future = publisher.publish(topic_path, msg_bytes)
    print(f"Published message ID: {future.result()}")
else:
    print("[stub] Set GCP credentials + PUBSUB_TOPIC to publish live.")
    print("  Pattern: publisher.publish(topic_path, build_ingestion_message(payload, envelope))")

---
## Section 5 — Google ADK: agent loop with HKI callbacks

The orchestrator service uses Google ADK as the agent executor. Two HKI callbacks
intercept every agent turn and every tool call:

- `HkiBeforeAgentCallback` — validates the envelope at session start; blocks the
  entire agent turn if the envelope is missing, expired, or in a forbidden domain
- `HkiBeforeToolCallback` — enforces domain on every tool invocation; blocks tool
  calls whose declared domain does not match the active domain
- `HkiToolGuard` — wraps a callable tool; enforces envelope before the function body runs

In [ ]:
import hki_adk

PHARMACY_ENVELOPE = make_envelope("pharmacy", purpose="chat")

# ── Simulate ADK callback context ────────────────────────────────────────────
class _FakeCallbackContext:
    """Minimal duck-type of google.adk.agents.CallbackContext"""
    def __init__(self, state: dict):
        self.state = state

class _FakeToolContext:
    """Minimal duck-type of google.adk.agents.ToolContext"""
    def __init__(self, state: dict, tool_name: str, tool_domain: str):
        self.state = state
        self.tool_name = tool_name
        self.tool_domain = tool_domain

# ── HkiBeforeAgentCallback ────────────────────────────────────────────────────
before_agent = hki_adk.HkiBeforeAgentCallback()

# Valid envelope in session state → allowed
ctx_ok = _FakeCallbackContext(state={"hki_envelope": PHARMACY_ENVELOPE})
try:
    before_agent(ctx_ok)
    print("Agent turn with pharmacy envelope → allowed")
except Exception as e:
    print(f"ERROR: {e}")

# No envelope → blocked
ctx_bad = _FakeCallbackContext(state={})
try:
    before_agent(ctx_bad)
    print("ERROR: should have been blocked")
except PermissionError as e:
    print(f"No envelope → blocked: {e}")

In [ ]:
# ── HkiBeforeToolCallback ─────────────────────────────────────────────────────
before_tool = hki_adk.HkiBeforeToolCallback()

class _FakeTool:
    def __init__(self, name, domain):
        self.name   = name
        self.domain = domain  # declared domain on the tool registration

# Tool in the correct domain → allowed
ctx_tool_ok = _FakeToolContext(
    state={"hki_envelope": PHARMACY_ENVELOPE},
    tool_name="rx.lookup",
    tool_domain="pharmacy",
)
try:
    before_tool(ctx_tool_ok, tool=_FakeTool("rx.lookup", "pharmacy"), args={}, kwargs={})
    print("Tool rx.lookup (pharmacy domain) → allowed")
except Exception as e:
    print(f"Unexpected block: {e}")

# Tool in a different domain → blocked
ctx_tool_bad = _FakeToolContext(
    state={"hki_envelope": PHARMACY_ENVELOPE},
    tool_name="hotel.search",
    tool_domain="travel",
)
try:
    before_tool(ctx_tool_bad, tool=_FakeTool("hotel.search", "travel"), args={}, kwargs={})
    print("ERROR: cross-domain tool should have been blocked")
except PermissionError as e:
    print(f"Tool hotel.search (travel) from pharmacy envelope → blocked: {e}")

In [ ]:
# ── HkiToolGuard: wrap a function as an HKI-enforced ADK tool ─────────────────
def raw_search_tool(query: str, tool_context=None) -> dict:
    """Simulates a knowledge search tool."""
    return {"results": [f"result for '{query}'"]}

# Wrap it — the guard validates the envelope before raw_search_tool() executes
safe_search_tool = hki_adk.HkiToolGuard(raw_search_tool)

# Call with a valid context
ctx_with_envelope = _FakeCallbackContext(state={"hki_envelope": PHARMACY_ENVELOPE})
try:
    result = safe_search_tool("return policy", tool_context=ctx_with_envelope)
    print(f"Safe tool result: {result}")
except Exception as e:
    print(f"Blocked: {e}")

# Call without context → blocked
ctx_no_envelope = _FakeCallbackContext(state={})
try:
    safe_search_tool("return policy", tool_context=ctx_no_envelope)
except PermissionError as e:
    print(f"No envelope on tool_context → blocked: {e}")

In [ ]:
# ── Wiring it into a real ADK agent (requires google-adk) ────────────────────
if LIVE_GCP:
    try:
        import google.adk.agents as adk_agents
        import google.adk.runners as adk_runners
        import google.adk.sessions as adk_sessions

        # This is the pattern from orchestrator-service/src/core/adk_agent.py
        agent = adk_agents.Agent(
            name="pharmacy-agent",
            model="gemini-2.0-flash",
            instruction="You are a pharmacy support agent. Only answer pharmacy questions.",
            tools=[safe_search_tool],
            before_agent_callback=hki_adk.HkiBeforeAgentCallback(),
            before_tool_callback=hki_adk.HkiBeforeToolCallback(),
        )

        session_service = adk_sessions.InMemorySessionService()
        runner = adk_runners.Runner(
            agent=agent,
            app_name="pharmacy-demo",
            session_service=session_service,
        )
        print("ADK agent configured with HKI callbacks")
    except ImportError:
        print("google-adk not installed — pip install google-adk")
else:
    print("[stub] ADK agent wiring pattern:")
    print("  agent = adk_agents.Agent(")
    print("      before_agent_callback=hki_adk.HkiBeforeAgentCallback(),")
    print("      before_tool_callback=hki_adk.HkiBeforeToolCallback(),")
    print("  )")

---
## Section 6 — Document AI + GCS: domain-labelled ingestion

Every document ingested through Document AI is labelled with the domain from
the signed envelope at the time of upload. The label is stored both in GCS
object metadata and in the AlloyDB `domain` column.

**Why this matters:** A document uploaded in a pharmacy session must never become
visible to travel agents — even if both use the same GCS bucket. The domain label
is the only thing that makes the isolation verifiable.

In [ ]:
# ── Document ingestion pipeline with HKI labelling ───────────────────────────
# Mirrors ingestion-pipeline-service/src/document_ai.py and gcs_store.py

import hashlib

def ingest_document(
    content: bytes,
    filename: str,
    envelope: dict,
) -> dict:
    """
    Ingest a document with domain labelling enforced by the HKI envelope.
    In production this calls Document AI then stores in GCS + AlloyDB.
    """
    # Step 1: validate envelope first — fail before spending any compute
    result = hki_runtime.validate_envelope(envelope, require_signature=True)
    if not result.ok:
        raise PermissionError(f"ingestion refused: {[i.message for i in result.issues]}")
    env = result.envelope

    # Step 2: extract text (Document AI in production)
    # real: docai.ProcessRequest(name=processor_name, raw_document=RawDocument(content, mime))
    extracted_text = content.decode(errors="ignore")  # stub

    # Step 3: build the artifact with domain label from envelope
    doc_id   = hashlib.sha256(content).hexdigest()[:16]
    gcs_path = f"{env.org_id}/{env.active_domain}/{doc_id}/{filename}"  # domain in path

    artifact = {
        "id":       doc_id,
        "org_id":   env.org_id,
        "domain":   env.active_domain,   # ← from signed envelope, not from filename/metadata
        "filename": filename,
        "gcs_path": gcs_path,
        "text":     extracted_text[:200],
        "envelope_id": env.envelope_id,  # audit trail
    }

    # Step 4: GCS metadata would include domain as a custom attribute
    gcs_metadata = {
        "hki-domain":      env.active_domain,
        "hki-org-id":      env.org_id,
        "hki-envelope-id": env.envelope_id,
    }

    return {"artifact": artifact, "gcs_metadata": gcs_metadata}

# Ingest as pharmacy domain
result = ingest_document(
    content=b"Pharmacy policy: Medications may only be dispensed with valid prescription.",
    filename="pharmacy-policy-2026.pdf",
    envelope=make_envelope("pharmacy", purpose="ingest"),
)
print(f"GCS path:   {result['artifact']['gcs_path']}")
print(f"Domain:     {result['artifact']['domain']}")
print(f"Envelope:   {result['artifact']['envelope_id']}")
print(f"GCS labels: {result['gcs_metadata']}")

In [ ]:
# ── Live GCS upload (requires GOOGLE_APPLICATION_CREDENTIALS + GCS_BUCKET) ───
GCS_BUCKET = os.environ.get("GCS_BUCKET", "")

if LIVE_GCP and GCS_BUCKET:
    from google.cloud import storage as gcs

    client = gcs.Client(project=GCP_PROJECT)
    bucket = client.bucket(GCS_BUCKET)

    result = ingest_document(
        content=b"Live pharmacy document",
        filename="live-test.txt",
        envelope=make_envelope("pharmacy", purpose="ingest"),
    )

    blob = bucket.blob(result["artifact"]["gcs_path"])
    blob.metadata = result["gcs_metadata"]  # HKI domain labels as GCS metadata
    blob.upload_from_string(b"Live pharmacy document")
    print(f"Uploaded: gs://{GCS_BUCKET}/{result['artifact']['gcs_path']}")
    print(f"Labels:   {blob.metadata}")
else:
    print("[stub] Set GCS_BUCKET env var to upload live.")
    print("  Pattern: blob.metadata = {'hki-domain': env.active_domain, ...}")

---
## Section 7 — End-to-end: gateway → ADK → retrieval → Gemini

The full request path in the production HKI platform:

```
1. User sends chat request
       ↓
2. Gateway mints signed HKI envelope (active_domain = "pharmacy")
       ↓ X-HKI-Envelope header
3. HkiMiddleware validates envelope → binds to request.state.hki
       ↓
4. Orchestrator: HkiBeforeAgentCallback validates envelope in session state
       ↓
5. ADK tool call → HkiBeforeToolCallback checks tool.domain == "pharmacy"
       ↓
6. Knowledge API: SQL WHERE domain='pharmacy' + assert_artifact_visible()
       ↓
7. LLM call: HkiLiteLLMCallback stamps cache key, validates envelope
       ↓ vertex_ai/gemini-2.0-flash
8. Response returned — no travel/membership/warehouse data visible
```

Every arrow is enforced by an HKI primitive. No single point of failure.

In [ ]:
# ── Simulate the full request flow ───────────────────────────────────────────

import hki_litellm

litellm.callbacks = [hki_litellm.HkiLiteLLMCallback()]

def simulate_full_request(user_domain: str, question: str) -> dict:
    trace = []

    # Step 1-2: Gateway mints envelope
    env_dict = make_envelope(user_domain, purpose="chat")
    result   = hki_runtime.validate_envelope(env_dict)
    env      = result.envelope
    trace.append(f"[gateway]     envelope minted  active_domain={env.active_domain}")

    # Step 3: Middleware (HkiMiddleware in production FastAPI)
    scope_override = hki_runtime.reject_conflicting_scope_argument(env_dict, {"domain": "travel"})
    if scope_override is None:
        trace.append(f"[middleware]  envelope valid, no scope override")

    # Step 4: ADK before_agent_callback
    ctx = _FakeCallbackContext(state={"hki_envelope": env_dict})
    hki_adk.HkiBeforeAgentCallback()(ctx)
    trace.append(f"[adk-agent]   before_agent_callback passed")

    # Step 5: Tool call — knowledge search
    TOOL_DOMAIN = user_domain   # tool is registered for this domain
    decision = hki_runtime.evaluate_gateway_target(env_dict, {
        "type": "tool", "id": "knowledge.search", "domain": TOOL_DOMAIN,
    })
    trace.append(f"[adk-tool]    knowledge.search allowed={decision.allowed}")

    # Step 6: Knowledge retrieval — domain-scoped SQL
    visible_chunks = [
        c for c in CHUNK_STORE
        if c["org_id"] == env.org_id and c["domain"] == env.active_domain
        and hki_runtime.assert_artifact_visible(env, {"org_id": c["org_id"], "domain": c["domain"], "artifact_type": "chunk", "artifact_id": c["id"]}) is None
    ]
    context = " | ".join(c["content"] for c in visible_chunks)
    trace.append(f"[knowledge]   {len(visible_chunks)} chunks retrieved (domain={user_domain})")

    # Step 7: LLM call — pre_call stamps HKI cache key + validates envelope
    llm_kwargs = {
        "hki_envelope": env_dict,
        "model":        "vertex_ai/gemini-2.0-flash",
        "messages":     [{"role": "user", "content": f"Context: {context}\n\n{question}"}],
        "metadata":     {},
    }
    hki_litellm.pre_call(llm_kwargs)
    cache_key = llm_kwargs["metadata"]["hki_cache_key"]
    trace.append(f"[litellm]     cache_key={cache_key[:24]}...")

    return {"trace": trace, "context_used": context, "cache_key": cache_key}

print("=== PHARMACY REQUEST ===")
r = simulate_full_request("pharmacy", "What are the prescription pickup hours?")
for line in r["trace"]:
    print(f"  {line}")
print(f"  Context sent to LLM: {r['context_used'][:80]}...")

print()
print("=== TRAVEL REQUEST (same question, different domain) ===")
r2 = simulate_full_request("travel", "What are the prescription pickup hours?")
for line in r2["trace"]:
    print(f"  {line}")
print(f"  Context sent to LLM: {r2['context_used'][:80] or '(empty — no pharmacy docs in travel domain)'}")

print()
print(f"Cache keys are different: {r['cache_key'][:20] != r2['cache_key'][:20]}")

In [ ]:
# ── Full live request (requires GCP credentials) ───────────────────────────────
if LIVE_GCP:
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["VERTEXAI_PROJECT"]  = GCP_PROJECT
    os.environ["VERTEXAI_LOCATION"] = GCP_LOCATION

    # The real call — envelope enforced by HkiLiteLLMCallback
    try:
        response = litellm.completion(
            model="vertex_ai/gemini-2.0-flash",
            messages=[
                {"role": "system", "content": "You are a pharmacy support agent."},
                {"role": "user",   "content": "What is the return policy for medications?"},
            ],
            metadata={"hki_envelope": make_envelope("pharmacy", purpose="chat")},
        )
        print(f"Gemini response: {response.choices[0].message.content[:200]}")
    except Exception as e:
        print(f"LLM error (check credentials): {e}")
else:
    print("[stub] Final litellm.completion() call pattern:")
    print("  litellm.completion(")
    print("      model='vertex_ai/gemini-2.0-flash',")
    print("      messages=[...],")
    print("      metadata={'hki_envelope': make_envelope('pharmacy', purpose='chat')},")
    print("  )")
    print("  → HkiLiteLLMCallback intercepts, validates envelope, stamps cache key")

---
## Summary — GCP service map

| GCP Service | HKI integration point | Package | What it enforces |
|-------------|----------------------|---------|------------------|
| Cloud Run / GKE gateway | `HkiMiddleware` | `hki-runtime` | Envelope validation on every HTTP request |
| Vertex AI / Gemini | `HkiLiteLLMCallback` | `hki-litellm` | Envelope required before model call; domain-bound cache key |
| AlloyDB + pgvector | SQL `WHERE domain=$1` + `assert_artifact_visible` | `hki-runtime` | Domain isolation at retrieval |
| Cloud Pub/Sub | Envelope in message body | `hki-runtime` | Domain preserved across async job boundaries |
| Google ADK | `HkiBeforeAgentCallback`, `HkiBeforeToolCallback`, `HkiToolGuard` | `hki-adk` | Domain enforced at every agent turn and tool call |
| GCS | `hki-domain` object metadata + path prefix | `hki-runtime` | Document labelled at ingestion, auditable |
| Document AI | Envelope validated before extraction | `hki-runtime` | Domain assigned from envelope, not from filename |

### Environment variables

```bash
# Authentication (all services use Application Default Credentials)
GOOGLE_APPLICATION_CREDENTIALS=/path/to/service-account.json  # local dev only
# In GKE: Workload Identity — no key file needed

# Required by all services
GOOGLE_CLOUD_PROJECT=your-project-id
GCP_LOCATION=us-central1

# Knowledge API
ALLOYDB_URL=postgresql+asyncpg://user:pass@/db?host=/cloudsql/project:region:instance

# Ingestion pipeline
GCS_BUCKET=your-bucket-name
PUBSUB_TOPIC=ingestion-jobs

# LLM gateway
GOOGLE_GENAI_USE_VERTEXAI=true
VERTEXAI_PROJECT=your-project-id
VERTEXAI_LOCATION=us-central1
```

### Next notebooks

- [06 — AWS integration](./06_aws_integration.ipynb) — Bedrock + Aurora pgvector + SQS + EKS
- [07 — Conformance testing your adapter](./07_conformance.ipynb) — run all 28 cases against your system

**GitHub:** https://github.com/h3nok/HKI  
**Deployment guide:** [deploy/k8s/](../deploy/k8s/)  
**Architecture paper:** [docs/HKI-package/HERMETIC-KNOWLEDGE-ISOLATION.md](../docs/HKI-package/HERMETIC-KNOWLEDGE-ISOLATION.md)